In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST CONSTRUCTORS FILE
# ============================================================
# Purpose:
# 1. Read constructors.json using Spark DataFrame Reader API
# 2. Apply an explicit schema
# 3. Add ingestion metadata
# 4. Write the data into a Bronze Delta table
# ============================================================


# COMMAND ----------
# Load Environment Configuration

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------
# Load Bronze Helper Functions

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# Source file location

source_file = f"{landing_folder_path}/constructors.json"

# Target Bronze table

table_name = (
    f"{catalog_name}.{schema_name}.{bronze_prefix}_constructors"
)

print("Source File :", source_file)
print("Target Table:", table_name)


# COMMAND ----------

# ============================================================
# Step 1 - Define schema for constructors.json
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

constructors_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])


# COMMAND ----------

# ============================================================
# Step 2 - Read constructors.json
# ============================================================

constructors_df = (
    spark.read
    .format("json")
    .schema(constructors_schema)
    .load(source_file)
)

display(constructors_df)


# COMMAND ----------

# Check DataFrame schema

constructors_df.printSchema()


# COMMAND ----------

# ============================================================
# Step 3 - Add Ingestion Metadata
# ============================================================

constructors_final_df = add_ingestion_metadata(constructors_df)

display(constructors_final_df)


# COMMAND ----------

# ============================================================
# Step 4 - Write to Bronze Delta Table
# ============================================================

(
    constructors_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)


# COMMAND ----------

# ============================================================
# Step 5 - Validate Bronze Delta Table
# ============================================================

display(
    spark.table(table_name)
)